# Week 1 (continued) — Near-Duplicate Detection (MinHash + LSH)

**Goal:** find essays that are near-identical (light paraphrases, minor edits, regenerated copies) that exact-match dedup in `01_eda_baseline.ipynb` can't catch. The output is a `dup_cluster` id per essay, which becomes a hard grouping constraint for the leakage-safe train/val/test split — no cluster may be split across sets.

In [3]:
import pandas as pd
from datasketch import MinHash, MinHashLSH

## Setup: load data, reapply the Week 1 short-essay filter

Same as the end of the EDA notebook: load the data, compute word count, and drop the 2 degenerate short essays (corrupted/truncated generations) identified there before doing anything else with this data.

In [4]:
df = pd.read_csv('../data/raw/train.csv')
df['n_words'] = df['text'].str.split().str.len()

In [5]:
df = df[df['n_words'] >= 20].reset_index(drop=True)
print(df.shape)

(44866, 6)


## Shingling: turning essays into comparable word-sequence sets

A **shingle** is a sliding window of `k` consecutive words. Two essays that share many shingles overlap heavily as sequences of words, even if individual sentences were reworded — this is what lets near-duplicate detection catch paraphrases that an exact string match misses. `k=5` is the standard choice for essay-length text.

In [6]:
def get_shingles(text, k=5):
    words = text.lower().split()
    return set(' '.join(words[i:i+k]) for i in range(len(words) - k + 1))

In [7]:
sample_shingles = get_shingles(df['text'].iloc[0])
print(len(sample_shingles))
list(sample_shingles)[:5]

375


['there is either an accident',
 'or tweet that someone sent.',
 'our generation. driving is one',
 'best way to come over',
 'and just have group chats']

## MinHash: compressing shingle sets into fixed-size signatures

Comparing full shingle sets pairwise across 44,866 essays is computationally impractical (~1 billion comparisons). MinHash sidesteps this: applying many hash functions to a shingle set and keeping only the minimum value from each produces a fixed-size signature (128 numbers here) whose agreement across two essays is an unbiased estimate of their true Jaccard similarity — without ever computing the full intersection.

In [8]:
def build_minhash(shingles, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for s in shingles:
        m.update(s.encode('utf8'))
    return m

**Sanity check 1** — identical text compared to itself should score essentially 1.0:

In [9]:
m1 = build_minhash(get_shingles(df['text'].iloc[0]))
m2 = build_minhash(get_shingles(df['text'].iloc[0]))
print(m1.jaccard(m2))

1.0


**Sanity check 2** — two different essays should score low. (Rows 0 and 1 are both "Phones and driving" essays but share no 5-word phrase, hence 0.0 — a useful sign that the 5-word shingle bar is meaningfully strict, not something that flags every same-topic essay.)

In [10]:
m3 = build_minhash(get_shingles(df['text'].iloc[1]))
print(m1.jaccard(m3))

0.0


## Building the LSH index (subset first)

Even comparing all pairs of 128-number signatures is still quadratic. **Locality-Sensitive Hashing (LSH)** avoids this by bucketing signatures so that only genuinely similar pairs are ever likely to collide and get compared. `threshold=0.7` sets how similar two essays' estimated Jaccard score must be before LSH treats them as a candidate match. Tested here on the first 2,000 rows first, to check correctness and timing before committing to the full dataset.

In [11]:
import time

lsh = MinHashLSH(threshold=0.7, num_perm=128)

t0 = time.time()
minhashes = {}
subset = df.iloc[:2000]

for idx, row in subset.iterrows():
    mh = build_minhash(get_shingles(row['text']))
    minhashes[idx] = mh
    lsh.insert(str(idx), mh)

print(f"Built and inserted {len(minhashes)} in {time.time()-t0:.1f}s")

Built and inserted 2000 in 16.8s


Confirm retrieval actually works — querying essay 0 should return at least itself:

In [12]:
result = lsh.query(minhashes[0])
print(result)

['0']


## Union-Find: merging chained near-duplicate matches into clusters

If essay A matches B, and B matches C, A and C should end up in the same cluster even if they weren't directly compared as similar enough to each other. A **Union-Find** (disjoint-set) structure handles exactly this: repeatedly declare "these two belong together," then ask "what group does this ultimately belong to," with path compression keeping repeated lookups fast.

In [13]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path compression
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[rx] = ry

## The full run: MinHash + LSH across all 44,866 essays

This is the expensive step — built once, in 268.6s (~4.5 minutes) here. No need to re-run this in future sessions; the output (`dup_cluster`, saved at the end of this notebook) is what later notebooks should load instead.

In [14]:
lsh_full = MinHashLSH(threshold=0.7, num_perm=128)
minhashes_full = {}

t0 = time.time()
for idx, row in df.iterrows():
    mh = build_minhash(get_shingles(row['text']))
    minhashes_full[idx] = mh
    lsh_full.insert(str(idx), mh)
print(f"Built {len(minhashes_full)} in {time.time()-t0:.1f}s")

Built 44866 in 268.6s


## Query every essay and merge matches into duplicate clusters

Querying all 44,866 signatures against the index took only 1.2s — this is exactly the payoff LSH is built for, versus the ~4.5 minutes the build itself took. The result: **43,780 clusters for 44,866 essays**, meaning 1,086 essays are near-duplicates of something else already in the dataset. That's a real, non-trivial amount of near-duplication that exact-match dedup in the EDA notebook completely missed.

In [15]:
uf = UnionFind(len(df))

t0 = time.time()
for idx in df.index:
    for r in lsh_full.query(minhashes_full[idx]):
        r_idx = int(r)
        if r_idx != idx:
            uf.union(idx, r_idx)
print(f"Queried all in {time.time()-t0:.1f}s")

df['dup_cluster'] = [uf.find(i) for i in df.index]
print(df['dup_cluster'].nunique(), "clusters for", len(df), "essays")

Queried all in 1.2s
43780 clusters for 44866 essays


## Are these duplicates meaningful? Checking cluster composition

Before deciding what to do about the 1,086 near-duplicate essays, it matters *what kind* of duplication this is. Three questions: how are cluster sizes distributed (mostly pairs, or a few large clusters?), do any clusters mix human and AI labels (a much bigger deal than same-class duplication), and do any clusters span multiple generators (`source`)?

In [16]:
cluster_sizes = df['dup_cluster'].value_counts()
print(cluster_sizes.value_counts().sort_index())
print("Largest cluster size:", cluster_sizes.max())

count
1     42703
2      1075
3         1
10        1
Name: count, dtype: int64
Largest cluster size: 10


**Label purity** — does any cluster contain both a human (`label=0`) and an AI (`label=1`) essay? If this comes back 0, duplication is happening within a class only (e.g. a generator producing near-identical essays for similar prompts, or near-identical human submissions) — annoying for leakage, but conceptually simple. If it's nonzero, that's worth a sentence in the report on its own: it would mean some AI-generated essays are near-verbatim of a human source essay.

In [17]:
label_purity = df.groupby('dup_cluster')['label'].nunique()
n_mixed_label = (label_purity > 1).sum()
print(n_mixed_label, "clusters contain both human and AI essays")

2 clusters contain both human and AI essays


**Source purity** — do any clusters span multiple generators? This helps explain *why* the duplication happened (e.g. the same underlying essay regenerated by more than one model, versus one generator repeating itself).

In [18]:
source_purity = df.groupby('dup_cluster')['source'].nunique()
n_mixed_source = (source_purity > 1).sum()
print(n_mixed_source, "clusters span multiple sources")

1069 clusters span multiple sources


## Reading an actual near-duplicate cluster

Numbers only go so far — read one real cluster end to end to confirm this is genuine near-duplication and not a threshold that's too loose.

In [19]:
dup_cluster_ids = cluster_sizes[cluster_sizes > 1].index
example_cluster = dup_cluster_ids[0]
df[df['dup_cluster'] == example_cluster][['label', 'source', 'prompt_name', 'text']]

,label,source,prompt_name,text
24,0,persuade_corpus,Phones and driving,Everyday people die in car accidents because t...
373,0,persuade_corpus,Phones and driving,Phones and Driving\n\nEveryday people die in c...
415,0,persuade_corpus,Phones and driving,Everyday people die in car accidents because t...
460,0,persuade_corpus,Phones and driving,Everyday people die in car accidents because t...
461,0,persuade_corpus,Phones and driving,Phones and Driving\n\nEveryday people die in c...
591,0,persuade_corpus,Phones and driving,Phones & Driving\n\nAlthough cell phones have ...
599,0,persuade_corpus,Phones and driving,Everyday people die in car accidents because t...
788,0,persuade_corpus,Phones and driving,Opponents say that cell phones are good becaus...
861,0,persuade_corpus,Phones and driving,Everyday people die in car accidents because t...
954,0,persuade_corpus,Phones and driving,Everyday people die in car accidents because t...


## Saving the clustered dataset for the split notebook

Rather than dropping near-duplicates now — a decision that's hard to undo and easy to get wrong before seeing the label/source purity results above — the safer, more defensible move is to keep every row and carry `dup_cluster` forward as a **grouping key**. The split notebook groups on it directly: every essay sharing a `dup_cluster` id must land entirely on one side of train/val/test, never split across. This is the same principle as grouping on `source` or `prompt_name`, just at the individual-essay level instead of the generator or topic level.

In [20]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/train_deduped.csv', index=False)
print("Saved:", df.shape)

Saved: (44866, 7)


## Key takeaways for the split notebook

- 43,780 clusters for 44,866 essays — 1,086 essays are near-duplicates of something else in the dataset, missed entirely by the exact-match check in the EDA notebook.
- Every essay now carries a `dup_cluster` id in `data/processed/train_deduped.csv` — use it as a **hard grouping constraint** in the split: no cluster may be divided across train/val/test.
- Check the label-purity and source-purity results above before writing the split notebook's data card — note explicitly whether any near-duplicate cluster crosses the human/AI boundary, since that's a finding worth reporting on its own regardless of the split mechanics.
- Combined grouping keys going into the split: `dup_cluster` (hard constraint, essay-level), `source` (for a held-out-generator robustness slice), `prompt_name` (for a held-out-topic slice).